# Strategy Router — query taxonomy demo

Three beats:

1. **Live extraction** — one call, three engines cooperating (regex + GLiNER2 + spaCy) with claim resolution.
2. **Corpus fingerprints** — same extractor, three retrieval corpora, three completely different profiles.
3. **The self-audit** — the profiler discovered a taxonomy gap on SciFact. Building the dataset improves the tool that builds the dataset.

> Run the first import cell BEFORE the meeting — `engines=None` triggers the spaCy + GLiNER2 model load (~15s cold).

## Beat 1 — live extraction

In [1]:
# Warm the extractor. engines=None = every engine (regex + GLiNER2 + spaCy).
# First call loads the pinned spaCy pipeline and the GLiNER2 checkpoint;
# subsequent calls are cheap (shared lru_cache).
from query_taxonomy.features import FeatureExtractor

extractor = FeatureExtractor(engines=None)
print("loaded — banks per group:")
for group, banks in extractor._by_group.items():
    engines = sorted({b.engine.value for b in banks})
    print(f"  {group.value:<24} {len(banks):>2} banks  engines: {engines}")

loaded — banks per group:
  structured_identifiers   82 banks  engines: ['gliner_model', 'regex']
  sentence_markers          6 banks  engines: ['regex']
  logical_structures        3 banks  engines: ['gliner_model', 'regex']
  corruption                1 banks  engines: ['regex']
  statistical_metrics       5 banks  engines: ['regex', 'spacy_model']


In [2]:
# One handpicked query — every engine has something to say about it.
#
# regex  : v2.1.0 (semver), 192.168.0.1 (ipv4), $750 (currency)
# gliner : "Dr. Smith" (person), "Berlin" (location)
# regex  : "October" (temporal, deterministic branch), "before" (negation-adjacent marker)
# spacy  : POS histogram, morphology, syntactic depth
#
# Claim registry ensures NUMBER can't steal `2.1.0` out of `v2.1.0`.

query = (
    "How do I upgrade to v2.1.0 on 192.168.0.1 before October in Berlin, "
    "per Dr. Smith's $750 quote?"
)

features = extractor.resolve(query)

print(f"query: {query}\n")
print("SPANS (claim-resolved, by group):")
for group, by_type in features.spans.items():
    print(f"  == {group.value}")
    for type_, matches in by_type.items():
        forms = ", ".join(m.text for m in matches)
        print(f"     {type_:<14} -> {forms}")

print("\nSTATS (spaCy, selected):")
metrics = features.stats.get(next(iter(features.stats), None), {})
for type_, stats in metrics.items():
    for stat in stats:
        if stat.name in {
            "length_tokens", "parse_depth", "clause_count",
            "closed_class_share", "inflected_share",
        }:
            print(f"     {type_}.{stat.name:<20} = {stat.value:.3f}")

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
query: How do I upgrade to v2.1.0 on 192.168.0.1 before October in Berlin, per Dr. Smith's $750 quote?

SPANS (claim-resolved, by group):
  == structured_identifiers
     ip_address     -> 192.168.0.1
     currency_amount -> $750
     version_string -> v2.1.0
     person         -> Dr. Smith
     location       -> Berlin
  == logical_structures
     temporal       -> October

STATS (spaCy, selected):
     length.length_tokens        = 23.000
     pos_profile.closed_class_share   = 0.476
     morphology.inflected_share      = 0.000
     syntactic_depth.parse_depth          = 4.000
     syntactic_depth.clause_count         = 1.000


**What does every stat mean?**
- `length.length_tokens` - 23 tokens, more longer than short query
- `pos_profile.closed_class_share` - fraction of tokens whose UD POS tag is a closed class (ADP, AUX, CCONJ, DET, NUM, PART, PRON, SCONJ), i.e. function words like the, of, to, is, when, and. Router signal: this is the canonical "how natural-language-shaped is this query?" measure — a keyword telegram like "CVE-2024-3094 xz-utils" sits near 0.0, a proper English sentence sits near 0.4–0.5. 0.476 = fully natural NL.
- `morphology.inflected_share` - fraction of alphabetic tokens whose lemma differs from surface form (e.g. running → lemma run).  High inflection → dense embeddings help more (they abstract over morphology); zero inflection → BM25 will not be tripped up by conjugation. 0.000 means every content word was already in lemma form.
- `syntactic_depth.parse_depth` - max depth of the dependency parse tree (root = depth 0). Distance from the deepest token up to ROOT in spaCy's parser. Router signal: compositional structure a bag-of-words loses. A deep tree (nested clauses, modifiers of modifiers) is where dense wins clearly; shallow trees are safe for sparse. 4 is a mid-depth English question.
- `syntactic_depth.clause_count` - number of tokens whose dependency label is clausal. Router signal: counts how many propositions the query is really asking about. = 1 means a single-clause question; ≥ 2 means multi-part queries where re-ranking / decomposition often matters.

In [3]:
# Contrast set — three queries picked so the profile *shape* is obvious per
# strategy the router should learn:
#
#   1. dense-friendly    : natural language, few identifiers
#   2. sparse-friendly   : identifier-dense, low NL scaffolding
#   3. hybrid-friendly   : mixed — NL wrap + concrete keys

queries = [
    "why do cats knead blankets when they are happy",
    "CVE-2024-3094 xz-utils sshd 5.6.0 backdoor",
    "how to migrate Postgres 14 to 16 without downtime on AWS RDS",
]

for query in queries:
    f = extractor.resolve(query)
    span_summary = {
        type_: len(matches)
        for group in f.spans.values() for type_, matches in group.items()
    }
    metrics = next(iter(f.stats.values()), {})
    closed = next(
        (s.value for stats in metrics.values() for s in stats
         if s.name == "closed_class_share"),
        None,
    )
    depth = next(
        (s.value for stats in metrics.values() for s in stats
         if s.name == "parse_depth"),
        None,
    )
    print(f"query: {query}")
    print(f"  spans        : {span_summary or '{}'}")
    print(f"  closed_class : {closed:.3f}" if closed is not None else "  closed_class : n/a")
    print(f"  parse_depth  : {depth:.0f}" if depth is not None else "  parse_depth  : n/a")
    print()

query: why do cats knead blankets when they are happy
  spans        : {'proper_noun': 2}
  closed_class : 0.556
  parse_depth  : 2

query: CVE-2024-3094 xz-utils sshd 5.6.0 backdoor
  spans        : {'cve': 1, 'version_string': 1, 'proper_noun': 3, 'acronym': 1}
  closed_class : 0.111
  parse_depth  : 4

query: how to migrate Postgres 14 to 16 without downtime on AWS RDS
  spans        : {'stock_ticker': 2, 'number': 2, 'negation': 1, 'acronym': 2}
  closed_class : 0.583
  parse_depth  : 5



## Beat 2 — corpus fingerprints

Same extractor, three retrieval corpora (`TrecDL2022(30000)`, `NFCorpus`, `SciFact`), three completely different profiles. These summaries were pre-computed by `src/profile_datasets.py`.

In [4]:
from pathlib import Path

PROFILES = Path("data/profiles")

for name in ("trec-dl-2022", "nfcorpus", "scifact"):
    header = f"### {name}"
    print(header)
    print("=" * len(header))
    print((PROFILES / f"{name}.summary.txt").read_text())
    print()

### trec-dl-2022
queries: 76 tagged: 72 (94.7%)

== structured_identifiers (4 types, 71 tagged)
-- entities (model) (3 types, 69 tagged)
proper_noun  queries= 56 matches= 64 diversity= 64
             top: temperature (1), business architect (1), ducane grill (1)
location     queries= 16 matches= 18 diversity= 18
             top: bahamas (1), westpac (1), three countries (1)
person       queries= 11 matches= 11 diversity= 11
             top: family advocate (1), auslan (1), gehan homes (1)
-- general (1 types, 3 tagged)
number  queries=  3 matches=  3 diversity=  3
        top: 8 (1), 1984 (1), 311 (1)

== logical_structures (1 types, 5 tagged)
temporal  queries=  5 matches=  5 diversity=  5
          top: october (1), 1984 (1), this year (1)

== sentence_markers (1 types, 1 tagged)
politeness  queries=  1 matches=  1 diversity=  1
            top: can you (1)

== statistical_metrics (5 types)
length.length_chars  docs=76 mean=34.974 min=13.000 max=72.000
length.length_tokens  docs=7

**What to notice, corpus by corpus:**

- **trec-dl-2022** — web queries. Clean natural language, ~6.5 tokens, closed-class share 0.42. Almost no structured identifiers (3 numbers total across 76 queries). This is the *dense-friendly* background.
- **nfcorpus** — health / nutrition Q&A. Proper-noun heavy (75% of queries), a few acronyms (ADHD, BMAA), negation markers on 11 queries. Comparative/NL scaffolding present.
- **scifact** — scientific claims. 98% tagged, identifier density explodes: 87 queries with acronym-shaped tokens, gene symbols like `TCR`, `IL-10`, `miRNA`. Character of a totally different retrieval regime.

## Beat 3 — the self-audit

The profiler doesn't just measure corpora — it audits its own taxonomy. Case in point on SciFact:

In [6]:
import json
from collections import Counter
from pathlib import Path

PROFILES = Path("data/profiles")
scifact = json.loads((PROFILES / "scifact.json").read_text())
identifiers = scifact["span_profiles"]["structured_identifiers"]

suspects = ("stock_ticker", "derivatives_symbol", "ticket", "booking_reference")
print("finance/logistics banks on SciFact — top surface forms:\n")
for name in suspects:
    if name not in identifiers:
        continue
    profile = identifiers[name]
    counts: Counter[str] = Counter()
    for spans in profile["spans"].values():
        for span in spans:
            # FeatureSpan is a NamedTuple -> serializes as [text, start, end]
            counts[span[0]] += 1
    top = counts.most_common(8)
    forms = ", ".join(f"{text} ({n})" for text, n in top)
    print(f"  {name:<20} ({sum(counts.values())} matches) -> {forms}")

finance/logistics banks on SciFact — top surface forms:

  stock_ticker         (116 matches) -> DNA (7), TCR (6), PPAR (4), IL (3), IR (3), II (3), UCB (3), UK (2)
  derivatives_symbol   (15 matches) -> ALDH1 (2), PIN1 (2), NF2 (2), PGAM1 (1), CHEK2 (1), IF3 (1), IRG1 (1), TIF2 (1)
  ticket               (5 matches) -> TDP-43 (2), IL-10 (2), CK-666 (1)
  booking_reference    (13 matches) -> CX3CR1 (4), SA1 (2), CD4 (1), EB1 (1), ND3 (1), ND6 (1), CD8 (1), PTENP1 (1)


**The finding:**

None of these are finance instruments. They're **gene/protein symbols and biochemical identifiers** — `TCR`, `IL-10`, `TDP-43`, `ALDH1`, `PIN1`, `CX3CR1`, `miRNA`, `iPSC`. Bare caps + digit shapes look identical to tickers/tickets, so the AMBIGUOUS-tier banks absorb them.

This is the point:

1. The FP smell test **worked** — off-topic-domain hits are a priori suspect, and the doctrine was written into `summary()`'s docstring before this run existed.
2. The FPs pointed at a **real taxonomy gap**: no `GENE_SYMBOL` bank. Scientific corpora have their own identifier family we hadn't modeled.
3. Filling that gap will **simultaneously de-noise finance banks on scientific corpora** — one bank does double duty.

> Profiling doesn't just tell us which corpora to sample — it tells us which banks to build next.

## Takeaways 

- **One extractor, three engines** (regex / GLiNER2 / spaCy), pluggable via `engines=` — same pipeline handles determinism-first identifiers and model-backed entities without the caller thinking about it.
- **Claim resolution is within-group** — layered banks compete for char ranges by ambiguity tier; groups stay independent layers.
- **The profile is the interface** to the diversified-dataset recipe: per-corpus fingerprints drive stratum design and harvest targets.
- **The taxonomy is not frozen.** Every profile run is a chance to spot a gap — SciFact just requested a `GENE_SYMBOL` bank.